In [ ]:
!pip install langchain_google_genai

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d043ed5caebabaf0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def greet(name):
    return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="textbox", outputs="textbox")

demo.launch(share=True)  # Kopīgo savu demo ar tikai 1 papildu parametru 🚀

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08e897893c7a3a0e70.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def analyze_sentiment(sentence):
   if "good" in sentence:
       return "positive"
   elif "bad" in sentence:
       return "negative"
   else:
       return "neutral"

with gr.Blocks() as app:
   sentence = gr.Textbox()
   sentiment = gr.Label()
   submit = gr.Button()

   submit.click(fn=analyze_sentiment,
                inputs=sentence,
                outputs=sentiment)

app.launch()

# Gradio Blocks

In [ ]:
import gradio as gr

def analyze_sentiment(sentence):
   if "good" in sentence:
       return "positive"
   elif "bad" in sentence:
       return "negative"
   else:
       return "neutral"

with gr.Blocks() as app:
   sentence = gr.Textbox()
   sentiment = gr.Label()
   submit = gr.Button()

   submit.click(fn=analyze_sentiment,
                inputs=sentence,
                outputs=sentiment)

app.launch()

# The Interface class

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * intensity

# If you use the actual classes for gr.Textbox and gr.Slider instead of the string shortcuts,
# you have access to much more customizability through component attributes.
demo = gr.Interface(
    fn=greet,
    inputs=["text", gr.Slider(value=2, minimum=1, maximum=10, step=1)],
    outputs=[gr.Textbox(label="greeting", lines=3)],
)

demo.launch()


In [ ]:
# Multiple Input and Output Components

import gradio as gr

def greet(name, is_morning, temperature):
    salutation = "Good morning" if is_morning else "Good evening"
    greeting = f"{salutation} {name}. It is {temperature} degrees today"
    celsius = (temperature - 32) * 5 / 9
    return greeting, round(celsius, 2)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "checkbox", gr.Slider(0, 100)],
    outputs=["text", "number"],
)
demo.launch()


In [ ]:
import numpy as np
import gradio as gr

def sepia(input_img):
    sepia_filter = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    sepia_img = input_img.dot(sepia_filter.T)
    sepia_img /= sepia_img.max()
    return sepia_img

demo = gr.Interface(sepia, gr.Image(), "image")
demo.launch()

In [ ]:
import gradio as gr

def calculator(num1, operation, num2):
    if operation == "add":
        return num1 + num2
    elif operation == "subtract":
        return num1 - num2
    elif operation == "multiply":
        return num1 * num2
    elif operation == "divide":
        if num2 == 0:
            raise gr.Error("Cannot divide by zero!")
        return num1 / num2

demo = gr.Interface(
    calculator,
    [
        "number",
        gr.Radio(["add", "subtract", "multiply", "divide"]),
        "number"
    ],
    "number",
    examples=[
        [45, "add", 3],
        [3.14, "divide", 2],
        [144, "multiply", 2.5],
        [0, "subtract", 1.2],
    ],
    title="Toy Calculator",
    description="Here's a sample toy calculator.",
)

demo.launch()

# Blocks and Event Listeners

In [ ]:
import gradio as gr


def greet(name):
    return "Hello " + name + "!"


with gr.Blocks() as demo:
    name = gr.Textbox(label="Name")
    output = gr.Textbox(label="Output Box")
    greet_btn = gr.Button("Greet")
    #  click() event listener
    greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

demo.launch()


In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    name = gr.Textbox(label="Name")
    output = gr.Textbox(label="Output Box")
    greet_btn = gr.Button("Greet")

    # event listener using decorator
    @greet_btn.click(inputs=name, outputs=output)
    def greet(name):
        return "Hello " + name + "!"

demo.launch()

# Chatbots

In [ ]:
import gradio as gr
import random

def random_response(message, history):
    return random.choice(["Yes", "No"])

gr.ChatInterface(
    fn=random_response,
    type="messages"
).launch()

In [ ]:
# Install dependencies (only needed in Colab)
# !pip install gradio langchain-google-genai

import gradio as gr
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from google.colab import userdata

# --- Initialize model ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GOOGLE_API_KEY
)

# --- Persistent chat history ---
chat_history = [SystemMessage(content="You are a helpful AI assistant")]

# --- Chat function for Gradio ---
def chat_with_ai(user_input, history):
    """
    user_input: latest message from the user
    history: list of previous [user, bot] messages from Gradio UI
    """
    # Convert Gradio history into LangChain message objects
    lc_messages = chat_history.copy()
    for user_msg, bot_msg in history:
        lc_messages.append(HumanMessage(content=user_msg))
        lc_messages.append(AIMessage(content=bot_msg))

    # Append the latest user message
    lc_messages.append(HumanMessage(content=user_input))

    # Invoke the model
    result = model.invoke(lc_messages)
    response = result.content

    # Append to history for display
    history.append((user_input, response))

    return history, history  # update chatbot display & keep state

# --- Gradio UI ---
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 Gemini Chatbot")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Type your message...")
    clear = gr.Button("Clear Chat")

    # State for conversation
    state = gr.State([])  # stores [(user, bot), ...]

    # When user sends a message
    msg.submit(chat_with_ai, [msg, state], [chatbot, state])
    msg.submit(lambda: "", None, msg)  # clear input box

    # Clear button resets everything
    clear.click(lambda: ([], []), None, [chatbot, state])

# Launch app
demo.launch()